In [2]:
import pandas as pd
import numpy as np
import difflib
from nltk.metrics.distance import *
from collections import defaultdict

Creating table of all KRAS alterations with hamming distances to be used for calculating probabilities.

In [3]:
def hamming_distance(s1, s2):
    # Handles case when strings aren't same length
    # Returns number of character changes to transform s1 to s1 (no rearrangements allowed)
    str_len = min([len(s1), len(s2)])
    extra_dist = max([len(s1), len(s2)]) - str_len
    
    return sum([s1[i] != s2[i] for i in range(str_len)]) + extra_dist

def calculate_complement(seq):
    complement_dict = {"A": "T", "T": "A", "G": "C", "C": "G"}
    return ''.join([complement_dict[i] for i in seq])

def dns_change(s1, s2):
    # Returns True/False for if s1 can be transformed to s2 via an adjacent change of length 2
    # Assumes s1 and s2 are the same length
    if (hamming_distance(s1, s2) == 2) and (len(s1) == 2) and (len(s2) == 2):
        return True
    else:
        if hamming_distance(s1, s2) == 2:
            return dns_change(s1[:-1], s2[:-1]) or dns_change(s1[1:], s2[1:])
        return False

In [4]:
delta = pd.read_csv("delta_nt.csv")
delta["Change"] = delta["Wt_codon"] + ":" + delta["Mut_codon"]
delta["Edit_Distance"] = delta.apply(lambda x: hamming_distance(x.Wt_codon, x.Mut_codon), axis=1)
delta["Is_DNS"] = delta.apply(lambda x: dns_change(x.Wt_codon, x.Mut_codon), axis=1)
delta.head()

,Index,Position,Wt_aa,Wt_codon,Mut_codon,Mut_aa,Mutation,Change,Edit_Distance,Is_DNS
0,1,2,T,ACT,GCT,A,T2A,ACT:GCT,1,False
1,2,2,T,ACT,GCC,A,T2A,ACT:GCC,2,False
2,3,2,T,ACT,GCA,A,T2A,ACT:GCA,2,False
3,4,2,T,ACT,GCG,A,T2A,ACT:GCG,2,False
4,5,2,T,ACT,TGT,C,T2C,ACT:TGT,2,True


Creating single base substitution probability table with keys matching format of sequence slices.

In [144]:
sbs = pd.read_csv("SBS_GRCh38_Cosmic_v3.3.csv")
sbs["From"] = sbs["Type"].apply(lambda x: x[0] + x[2] + x[6])
sbs["To"] = sbs["Type"].apply(lambda x: x[0] + x[4] + x[6])
sbs["SBS"] = sbs["From"] + ":" + sbs["To"]
del sbs["Type"]

sbs = sbs.loc[:, ["SBS"] + list(sbs.columns.values[:-3])]
sbs.set_index("SBS", inplace=True)
sbs.head()

,SBS1,SBS2,SBS3,SBS4,SBS5,SBS6,SBS7a,SBS7b,SBS7c,SBS7d,...,SBS85,SBS86,SBS87,SBS88,SBS89,SBS90,SBS91,SBS92,SBS93,SBS94
SBS,,,,,,,,,,,,,,,,,,,,,
ACA:AAA,0.000876,5.790000e-07,0.020920,0.042451,0.012052,0.000425,0.000067,0.002344,0.004841,0.000040,...,0.006108,0.002968,0.008946,1.000000e-18,0.032297,0.002222,0.002934,0.011396,0.011628,0.015677
ACC:AAC,0.002220,1.455050e-04,0.016343,0.032990,0.009337,0.000516,0.000177,0.000457,0.001135,0.000754,...,0.000871,0.003735,0.004490,1.000000e-18,0.017495,0.000704,0.052013,0.009653,0.008011,0.024523
ACG:AAG,0.000180,5.360000e-05,0.001808,0.016116,0.001908,0.000053,0.000073,0.000192,0.000388,0.000257,...,0.000316,0.000398,0.006357,1.000000e-18,0.009971,0.000144,0.000209,0.004851,0.001817,0.001627
ACT:AAT,0.001265,9.760000e-05,0.012265,0.029663,0.006636,0.000180,0.000249,0.000714,0.001964,0.004051,...,0.002728,0.003639,0.004941,1.737757e-03,0.020818,0.001771,0.000130,0.007800,0.008457,0.011141
ACA:AGA,0.001839,2.230000e-16,0.019813,0.006931,0.010144,0.000471,0.000065,0.000009,0.001123,0.001181,...,0.007268,0.052763,0.007843,1.000000e-18,0.014876,0.000513,0.000242,0.003074,0.008898,0.007048


KRAS sequence. Need to at G to the beginning (last NT in start codon) and T at the end (first NT in the stop codon).

In [145]:
kras_sequence = ''.join(delta.loc[:, ["Position", "Wt_codon"]].drop_duplicates()["Wt_codon"].values)

delta_seq = "G" + kras_sequence + "T"

In [146]:
# Listing all codon frames around each nucleotide in each KRAS codon
codon_rfs = [[delta_seq[i:i+3] for i in range(j, j+3)] for j in range(0, len(delta_seq)-2, 3)]

##### Algorithm Ideas
- If there is only one nucleotide that needs to change to create the aa change, it is treated as before where they are just saved and summed up at the end per signature (e.g. if T2A can happen 3 ways from one nucleotide change, then the probability of those three will be summed up at the end per signature)
- If there are two nucleotides that need to change to create the aa change, the probabilities of each single nucleotide change are multiplied and then saved and then summed at the end per signature (e.g. if T2A can happen 2 ways from two nucleotide changes, the probabilities of the two single nucleotide changes for the first way are multiplied and then summed with the product of the probabilities of the two single nucleotide changes for the second way)
- These are all summed up at the amino acid change level at the end to get an overall probability for aa changing

In [147]:
kras_mutational_prob = defaultdict(list)

for codon_position in delta["Position"].unique():
    
    position_delta = delta.loc[delta["Position"] == codon_position, :]
    position_delta = position_delta.loc[(position_delta["Edit_Distance"] != 3) & (~position_delta["Is_DNS"]), :]
    wt_codon = position_delta["Wt_codon"].values[0] # Always 1 unique value
    position_codon_rfs = codon_rfs[codon_position-2] # Getting frames around each nucleotide

    for mutation in position_delta["Mut_codon"].values:
        aa_change = position_delta.loc[position_delta["Mut_codon"] == mutation, "Mutation"].values[0]
        
        position_of_change = [j for j in range(len(wt_codon)) if wt_codon[j] != mutation[j]]

        # Case when two non-adjacent nucleotides change
        if len(position_of_change) > 1:
            dnp_mutational_prob = []
        
        for poc in position_of_change:
            mut_codon_frame = list(position_codon_rfs[poc])
            mut_codon_frame[1] = mutation[poc] # Changing middle nucleotide to mutant
            
            # COSMIC is indexed by WT Codon:Mutant Codon
            codon_change = f"{position_codon_rfs[poc]}:{''.join(mut_codon_frame)}"
            
            try:
                probability = sbs.loc[codon_change, :]
            except KeyError:
                # Complement is in sbs instead of standard
                wt_codon_comp = calculate_complement(position_codon_rfs[poc])
                mut_codon_comp = calculate_complement(''.join(mut_codon_frame))
                
                # Complements need to be reversed since strand direction is opposite
                comp_codon_change = f"{wt_codon_comp[::-1]}:{mut_codon_comp[::-1]}"
                
                probability = sbs.loc[comp_codon_change, :]
            
            # Adding probabilities to amino acid change
            if len(position_of_change) == 1:
                kras_mutational_prob[aa_change].append(probability)
            else:
                # Need to multiply probabilities if it requires two nucleotide changes (so log it so we can add them later which is same as multiplying)
                dnp_mutational_prob.append(np.log(probability))
        
        if len(position_of_change) > 1:
            kras_mutational_prob[aa_change].append(np.exp(np.sum(dnp_mutational_prob, axis=0)))

In [148]:
mutational_probabilities = kras_mutational_prob.copy()

for aa_change in mutational_probabilities:
    # Summing per signature
    mutational_probabilities[aa_change] = np.sum(kras_mutational_prob[aa_change], axis=0)

In [149]:
mutational_probabilities = pd.DataFrame.from_dict(mutational_probabilities)
mutational_probabilities.index = sbs.columns.values
mutational_probabilities = mutational_probabilities.transpose()
mutational_probabilities.head()

,SBS1,SBS2,SBS3,SBS4,SBS5,SBS6,SBS7a,SBS7b,SBS7c,SBS7d,...,SBS85,SBS86,SBS87,SBS88,SBS89,SBS90,SBS91,SBS92,SBS93,SBS94
T2A,1.862395e-03,0.000027,0.005810,0.001225,0.008151,0.001107,0.000074,2.216417e-16,0.000422,0.002305,...,0.011574,0.006861,0.003868,3.452168e-03,0.001724,0.000092,0.000134,0.002747,0.002832,0.001155
T2I,9.468128e-03,0.001856,0.012165,0.004273,0.022086,0.021298,0.007436,1.086161e-02,0.007494,0.032911,...,0.004582,0.004595,0.010666,1.000000e-18,0.029742,0.000652,0.002317,0.016541,0.006537,0.016606
T2N,1.265053e-03,0.000098,0.012265,0.029663,0.006636,0.000180,0.000249,7.140510e-04,0.001964,0.004051,...,0.002728,0.003639,0.004941,1.737757e-03,0.020818,0.001771,0.000130,0.007800,0.008457,0.011141
T2P,2.181740e-16,0.000132,0.002411,0.000263,0.001755,0.000212,0.000120,3.208539e-04,0.001655,0.000022,...,0.000798,0.000687,0.001230,1.726084e-03,0.001331,0.000046,0.001375,0.000301,0.002725,0.000642
T2S,1.253135e-03,0.000221,0.025198,0.007307,0.013127,0.000742,0.000318,5.425098e-04,0.004197,0.003022,...,0.014076,0.092832,0.013063,9.233019e-03,0.013582,0.001101,0.000395,0.006107,0.018368,0.008677


In [150]:
mutational_probabilities.to_csv("COSMIC-v3-KRAS-Mutational-Probabilities-Including-DNT.tsv", sep="\t")

#### Performing Analysis Using Adjacent DNS

In [130]:
dbs = pd.read_csv("DBS_GRCh38_Cosmic_v3.3.csv")

dbs["From"] = dbs["Type"].apply(lambda x: x[0:2])
dbs["To"] = dbs["Type"].apply(lambda x: x[3:])
dbs["DBS"] = dbs["From"] + ":" + dbs["To"]
del dbs["Type"]

dbs = dbs.loc[:, ["DBS"] + list(dbs.columns.values[:-3])]
dbs.set_index("DBS", inplace=True)
dbs.head()

,DBS1,DBS2,DBS3,DBS4,DBS5,DBS6,DBS7,DBS8,DBS9,DBS10,DBS11
DBS,,,,,,,,,,,
AC:CA,0.000050,3.776320e-04,0.006338,4.146665e-03,2.141988e-03,0.002665,0.045932,0.215132,0.024656,0.009590,0.001377
AC:CG,0.000027,3.200000e-05,0.000046,4.970000e-05,4.624290e-04,0.000146,0.006254,0.000974,0.002075,0.000018,0.000195
AC:CT,0.004331,1.272240e-04,0.003014,8.330000e-18,2.021876e-03,0.000240,0.013609,0.318195,0.028565,0.013443,0.000760
AC:GA,0.000128,2.393020e-04,0.005839,4.407083e-03,5.585180e-04,0.008597,0.009737,0.001161,0.025357,0.005328,0.000444
AC:GG,0.000058,4.210000e-18,0.000449,7.020000e-18,1.110000e-17,0.000104,0.002542,0.000810,0.007146,0.000588,0.002262


In [131]:
# Change is always internal (by def) so only have 2 windows per codon (since they are next to each other)
codon_rfs_dbs = [[delta_seq[i:i+2] for i in range(j, j+2)] for j in range(1, len(delta_seq)-2, 3)]

In [142]:
kras_mutational_prob_dbs = defaultdict(list)
not_present = []

for codon_position in delta["Position"].unique():
    
    position_delta = delta.loc[delta["Position"] == codon_position, :]
    position_delta = position_delta.loc[(position_delta["Edit_Distance"] == 2) & (position_delta["Is_DNS"]), :]
    wt_codon = position_delta["Wt_codon"].values[0] # Always 1 unique value
    position_codon_rfs = codon_rfs_dbs[codon_position-2] # Getting frames in codon
    
    for mutation in position_delta["Mut_codon"].values:
        aa_change = position_delta.loc[position_delta["Mut_codon"] == mutation, "Mutation"].values[0]    
        
        # Values in this should be consecutive
        position_of_change = [j for j in range(len(wt_codon)) if wt_codon[j] != mutation[j]]
        
        # COSMIC is indexed by WT Codon:Mutant Codon
        codon_change = f"{wt_codon[position_of_change[0]:position_of_change[1]+1]}:{mutation[position_of_change[0]:position_of_change[1]+1]}"
        
        try:
            probability = dbs.loc[codon_change, :]
        except KeyError:
                # Complement is in dbs instead of standard (also needs to be reversed since strand is reverse)
                wt_codon_comp = calculate_complement(wt_codon[position_of_change[0]:position_of_change[1]+1])
                mut_codon_comp = calculate_complement(mutation[position_of_change[0]:position_of_change[1]+1])
                
                comp_codon_change = f"{wt_codon_comp[::-1]}:{mut_codon_comp[::-1]}"
                
                probability = dbs.loc[comp_codon_change, :]
        
        if aa_change == "T20I":
            print(probability)
        # Adding probabilities to amino acid change
        kras_mutational_prob_dbs[aa_change].append(probability)

DBS1     0.000127
DBS2     0.005687
DBS3     0.007249
DBS4     0.000366
DBS5     0.002251
DBS6     0.018981
DBS7     0.000032
DBS8     0.000334
DBS9     0.000084
DBS10    0.007838
DBS11    0.000712
Name: CG:TT, dtype: float64
DBS1     1.290000e-05
DBS2     2.199228e-03
DBS3     7.299910e-04
DBS4     7.100000e-18
DBS5     3.020000e-05
DBS6     9.043951e-03
DBS7     4.940000e-05
DBS8     9.173810e-04
DBS9     1.350420e-04
DBS10    2.673356e-03
DBS11    6.975980e-04
Name: CG:TC, dtype: float64
DBS1     3.640610e-04
DBS2     4.270000e-18
DBS3     9.898354e-02
DBS4     7.100000e-18
DBS5     1.060000e-17
DBS6     8.420000e-18
DBS7     9.160000e-18
DBS8     5.129220e-04
DBS9     9.160000e-18
DBS10    3.611553e-01
DBS11    7.520000e-18
Name: CG:TA, dtype: float64


In [135]:
mutational_probabilities_dbs = kras_mutational_prob_dbs.copy()

for aa_change in mutational_probabilities_dbs:
    # Summing per signature
    mutational_probabilities_dbs[aa_change] = np.sum(kras_mutational_prob_dbs[aa_change], axis=0)
    
mutational_probabilities_dbs = pd.DataFrame.from_dict(mutational_probabilities_dbs)
mutational_probabilities_dbs.index = dbs.columns.values
mutational_probabilities_dbs = mutational_probabilities_dbs.transpose()
mutational_probabilities_dbs.head()

,DBS1,DBS2,DBS3,DBS4,DBS5,DBS6,DBS7,DBS8,DBS9,DBS10,DBS11
T2C,0.000174,6.420000e-06,1.110000e-17,5.320000e-05,1.121040e-03,0.000450,0.000999,0.002572,0.017941,0.000008,0.000240
T2D,0.000128,2.393020e-04,5.838502e-03,4.407083e-03,5.585180e-04,0.008597,0.009737,0.001161,0.025357,0.005328,0.000444
T2F,0.026751,2.948356e-03,1.780000e-06,1.612590e-04,7.476939e-03,0.003186,0.001181,0.000268,0.044501,0.000707,0.002835
T2G,0.000058,4.210000e-18,4.491160e-04,7.020000e-18,1.110000e-17,0.000104,0.002542,0.000810,0.007146,0.000588,0.002262
T2H,0.000050,3.776320e-04,6.337519e-03,4.146665e-03,2.141988e-03,0.002665,0.045932,0.215132,0.024656,0.009590,0.001377


In [136]:
mutational_probabilities_dbs.to_csv("COSMIC-v3-KRAS-Mutational-Probabilities-DBS.tsv", sep="\t")